# Automated Competitor Pricing Extraction

**Business Objective:** Extract dynamic pricing data from e-commerce platforms to monitor competitor price fluctuations and inform dynamic pricing models.

**Technical Approach:** Implementation of explicit waits (WebDriverWait) via Selenium to handle JavaScript-rendered DOM elements, followed by raw data parsing and cleansing using BeautifulSoup and Regular Expressions (Regex).

In [1]:
# Installation of Chrome and Selenium in Colab.
# Uncomment the following lines to setup the environment if running in Google Colab.

# !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# !apt-get install -y ./google-chrome-stable_current_amd64.deb
# !rm google-chrome-stable_current_amd64.deb
# !pip install selenium webdriver-manager unidecode

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
google-chrome-stable is already the newest version (146.0.7680.164-1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [2]:
# Import of libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import bs4
import re

In [3]:
# Configuration of Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')

In [4]:
# Browser initialization
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
driver.maximize_window()

In [5]:
# 1. PIPELINE CONFIGURATION
category_ids = ['20_26', '20_27', '24'] # Targets: PC, Mac, Phones
num_pages = 2

# Data storage (Using a list of dictionaries for Pandas optimization)
scraped_data = []

# 2. EXTRACTION LOOP
print("Initializing automated extraction pipeline...")

for cat_id in category_ids:
    for page in range(1, num_pages + 1):
        url = f"https://tutorialsninja.com/demo/index.php?route=product/category&path={cat_id}&page={page}"
        driver.get(url)

        # Explicit wait for dynamic DOM elements
        try:
            WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div.product-layout'))
            )
        except:
            continue # Skip page if elements fail to render

        # DOM Parsing
        soup = bs4.BeautifulSoup(driver.page_source, 'html5lib')
        cards = soup.find_all('div', attrs={'class': 'product-layout'})

        # Data Extraction & Cleansing
        for card in cards:
            # Extract Name
            product_name = card.find('div', attrs={'class':'caption'}).find('h4').find('a').text.strip()

            # Extract and Clean Price
            price_element = card.find('p', attrs={'class':'price'})
            if price_element:
                raw_price = price_element.text.split('Ex Tax')[0].strip()
                # Regex to remove non-numeric characters (except decimal point)
                clean_price = float(re.sub(r'[^\d.]', '', raw_price))
            else:
                clean_price = None

            # Append to dataset
            scraped_data.append({
                'Product_Name': product_name,
                'Price_USD': clean_price
            })

driver.close()

# 3. DATA STRUCTURING
df_pricing = pd.DataFrame(scraped_data)
print("Pipeline execution complete. Data structured successfully.")
df_pricing.head()

Initializing automated extraction pipeline...
Pipeline execution complete. Data structured successfully.


,Product_Name,Price_USD
0,iMac,122.00
1,HTC Touch HD,122.00
2,iPhone,123.20
3,Palm Treo Pro,337.99


In [7]:
# Save to CSV for downstream analytics
df_pricing.to_csv('ecommerce_pricing_data.csv', index=False)